# Stage 2 v2 local FP32 — ResNet-50 focal loss
Notebook chạy trực tiếp trên Windows/RTX 4050 bằng FP32. Chỉ chạy một notebook local tại một thời điểm; external test luôn bị khóa.

In [1]:
from pathlib import Path
import os, sys

# Nếu Jupyter không được mở từ repository, đặt ví dụ:
# LOCAL_REPO_ROOT = Path(r"D:\Junior\ĐATH\main")
LOCAL_REPO_ROOT = None

def discover_repo_root(explicit_root=None):
    candidates = []
    if explicit_root is not None:
        candidates.append(Path(explicit_root).expanduser().resolve())
    environment_root = os.environ.get("DATH_REPO_ROOT")
    if environment_root:
        candidates.append(Path(environment_root).expanduser().resolve())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    for candidate in candidates:
        if (candidate / "configs" / "default.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy repository. Hãy đặt LOCAL_REPO_ROOT bằng Path raw-string.")

REPO_DIR = discover_repo_root(LOCAL_REPO_ROOT)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

LOCAL_STAGE_ID = "stage2_focalloss_v2_local_fp32_resnet50"
MODEL_NAME = "resnet50"
MODEL_SLUG = "resnet50"
BASE_CONFIG = "configs/resnet50_aug_auprc.yaml"
WEIGHTING = "raw"
POSITIVE_WEIGHT = {"mode": "raw_ratio", "max_value": None}
RUN_FINAL_TEST = False
MAX_NEW_RUNS_PER_SESSION = None
TRAIN_BATCH_SIZE = 8
VALIDATION_BATCH_SIZE = 16
NUM_WORKERS = 0
PREFETCH_FACTOR = 2

PROCESSED_ROOT = REPO_DIR / "data" / "processed" / "consensus_v1_224"
PROTOCOL_ROOT = REPO_DIR / "data" / "protocols" / "vindr_s8515_exttest_v1"
CHECKPOINT_ROOT = REPO_DIR / "checkpoint" / LOCAL_STAGE_ID / MODEL_SLUG
LOCAL_STAGE_ROOT = REPO_DIR / "outputs" / LOCAL_STAGE_ID
RUN_LOG_ROOT = LOCAL_STAGE_ROOT / MODEL_SLUG
RUNS_ROOT = RUN_LOG_ROOT / "runs"
ANALYSIS_ROOT = RUN_LOG_ROOT / "analysis"
HARDWARE_ROOT = RUN_LOG_ROOT / "hardware"
RUNTIME_CONFIG_ROOT = LOCAL_STAGE_ROOT / "runtime_configs" / MODEL_SLUG
EXPORT_PATH = REPO_DIR / "outputs" / "exports" / "04_2_local_resnet50_stage2_focalloss_v2_fp32_results.zip"

assert RUN_FINAL_TEST is False
print("Repository:", REPO_DIR)
print("Processed data:", PROCESSED_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Run-log root:", RUN_LOG_ROOT)

Repository: D:\Junior\ĐATH\main
Processed data: D:\Junior\ĐATH\main\data\processed\consensus_v1_224
Checkpoint root: D:\Junior\ĐATH\main\checkpoint\stage2_focalloss_v2_local_fp32_resnet50\resnet50
Run-log root: D:\Junior\ĐATH\main\outputs\stage2_focalloss_v2_local_fp32_resnet50\resnet50


In [2]:
import json, platform, subprocess
import torch, torchvision
from torchvision.models import ResNet50_Weights

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if not torch.cuda.is_available():
    raise RuntimeError("Local Stage 2 yêu cầu CUDA GPU")
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_mib = int(torch.cuda.get_device_properties(0).total_memory / 1024 ** 2)
if "RTX 4050" not in gpu_name.upper():
    raise RuntimeError(f"Expected RTX 4050 Laptop GPU, got {gpu_name}")
if gpu_memory_mib < 5500:
    raise RuntimeError(f"Expected at least 5.5 GB VRAM, got {gpu_memory_mib} MiB")
if torch.__version__ != "2.7.0+cu126" or torchvision.__version__ != "0.22.0+cu126":
    raise RuntimeError(f"Use .venv-1 / requirements.txt; got torch={torch.__version__}, torchvision={torchvision.__version__}")
fp32_probe = torch.ones(8, 8, device="cuda", dtype=torch.float32) @ torch.ones(8, 8, device="cuda", dtype=torch.float32)
if not torch.isfinite(fp32_probe).all():
    raise RuntimeError("Local CUDA FP32 probe failed")
del fp32_probe
torch.cuda.empty_cache()
pretrained_state = ResNet50_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)
if not pretrained_state or not all(torch.isfinite(value).all().item() for value in pretrained_state.values() if torch.is_tensor(value)):
    raise RuntimeError("ResNet-50 pretrained state is missing or nonfinite")
del pretrained_state
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], check=True, capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()
hardware_profile = {
    "runtime_mode": "local_windows", "platform": platform.platform(),
    "cpu_count": os.cpu_count(), "gpu_name": gpu_name,
    "gpu_memory_mib": gpu_memory_mib, "nvidia_smi": smi,
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "train_batch_size": TRAIN_BATCH_SIZE, "validation_batch_size": VALIDATION_BATCH_SIZE,
    "num_workers": NUM_WORKERS, "prefetch_factor": PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    "amp": False, "precision": "float32",
    "cudnn_benchmark": False, "cudnn_deterministic": True,
}
print(json.dumps(hardware_profile, indent=2))
print("ResNet-50 pretrained + CUDA FP32 preflight: OK")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Test/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 69.1MB/s]


{
  "runtime_mode": "local_windows",
  "platform": "Windows-11-10.0.26300-SP0",
  "cpu_count": 16,
  "gpu_name": "NVIDIA GeForce RTX 4050 Laptop GPU",
  "gpu_memory_mib": 6140,
  "nvidia_smi": "NVIDIA GeForce RTX 4050 Laptop GPU, 6141, 610.62",
  "torch": "2.7.0+cu126",
  "torchvision": "0.22.0+cu126",
  "cuda": "12.6",
  "cudnn": 90701,
  "train_batch_size": 8,
  "validation_batch_size": 16,
  "num_workers": 0,
  "prefetch_factor": null,
  "amp": false,
  "precision": "float32",
  "cudnn_benchmark": false,
  "cudnn_deterministic": true
}
ResNet-50 pretrained + CUDA FP32 preflight: OK


In [3]:
import copy, yaml
from src.analysis.stage2 import analyze_stage2
from src.experiments.stage2_colab import _validate_train_data
from src.experiments.stage2_colab_v2 import (
    _run_completion, _run_subprocess_streaming, _start_telemetry, _stop_telemetry,
    build_stage2_experiments, create_or_validate_contract,
    validate_export_members, write_json_with_retry,
)
from src.utils.checksums import file_sha256
from src.utils.config import load_config

for path in (CHECKPOINT_ROOT, RUNS_ROOT, ANALYSIS_ROOT, HARDWARE_ROOT, RUNTIME_CONFIG_ROOT, EXPORT_PATH.parent):
    path.mkdir(parents=True, exist_ok=True)
data_summary = _validate_train_data(PROCESSED_ROOT, PROTOCOL_ROOT)
experiments = build_stage2_experiments(MODEL_NAME, MODEL_SLUG, BASE_CONFIG, WEIGHTING, POSITIVE_WEIGHT)
for experiment in experiments:
    experiment["run_name"] = experiment["run_name"].removesuffix("_amp") + "_fp32"
assert len(experiments) == 4 and len({item["run_name"] for item in experiments}) == 4

def build_local_runtime_config(experiment):
    config = load_config(REPO_DIR / experiment["config"])
    config["data"].update({
        "processed_root": str(PROCESSED_ROOT), "protocol_dir": str(PROTOCOL_ROOT),
        "train_manifest_path": str(PROTOCOL_ROOT / "train_manifest.csv"),
        "val_manifest_path": str(PROTOCOL_ROOT / "val_manifest.csv"),
    })
    config["training"].update({
        "batch_size": TRAIN_BATCH_SIZE, "validation_batch_size": VALIDATION_BATCH_SIZE,
        "num_workers": NUM_WORKERS, "persistent_workers": NUM_WORKERS > 0,
        "prefetch_factor": PREFETCH_FACTOR, "use_amp": False,
        "show_progress": False, "monitor": "macro_average_precision",
        "loss": copy.deepcopy(experiment["loss"]),
    })
    runtime_dir = RUNTIME_CONFIG_ROOT / experiment["run_name"]
    runtime_dir.mkdir(parents=True, exist_ok=True)
    runtime_path = runtime_dir / "default.yaml"
    runtime_path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return runtime_path, load_config(runtime_path)

runtime_configs = {item["run_name"]: build_local_runtime_config(item) for item in experiments}
for experiment in experiments:
    (CHECKPOINT_ROOT / experiment["run_name"]).mkdir(parents=True, exist_ok=True)
    result_dir = RUNS_ROOT / experiment["run_name"]
    result_dir.mkdir(parents=True, exist_ok=True)
    status_path = result_dir / "run_status.json"
    if not status_path.exists():
        write_json_with_retry(status_path, {"state": "pending", "run_name": experiment["run_name"]})
source_files = [REPO_DIR / "src/losses/classification_loss.py", REPO_DIR / "src/training/engine.py", REPO_DIR / "scripts/train.py", REPO_DIR / "src/experiments/stage2_colab_v2.py"]
source_sha256 = {str(path.relative_to(REPO_DIR)): file_sha256(path) for path in source_files}
contract = {
    "stage_id": LOCAL_STAGE_ID, "runtime_mode": "local_windows",
    "hardware_profile": hardware_profile, "data_summary": data_summary,
    "source_sha256": source_sha256, "experiments": experiments,
    "selection_metric": "macro_average_precision", "external_test_used": False,
}
create_or_validate_contract(LOCAL_STAGE_ROOT / "stage2_contract.json", contract, allow_create=True)
write_json_with_retry(HARDWARE_ROOT / "hardware_profile.json", hardware_profile)
write_json_with_retry(RUN_LOG_ROOT / "experiment_registry.json", experiments)
print([item["run_name"] for item in experiments])
print("LOCAL RESNET-50 STAGE 2 FP32 READY")

d:\Junior\ĐATH\main\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['resnet50_stage2v2_bce_raw_aug_auprc_s8515_exttest_v1_seed42_fp32', 'resnet50_stage2v2_focal_g05_raw_aug_auprc_s8515_exttest_v1_seed42_fp32', 'resnet50_stage2v2_focal_g10_raw_aug_auprc_s8515_exttest_v1_seed42_fp32', 'resnet50_stage2v2_focal_g15_raw_aug_auprc_s8515_exttest_v1_seed42_fp32']
LOCAL RESNET-50 STAGE 2 FP32 READY


In [4]:
import subprocess, traceback
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

new_runs = 0
for index, experiment in enumerate(experiments, start=1):
    config_path, config = runtime_configs[experiment["run_name"]]
    checkpoint_dir = CHECKPOINT_ROOT / experiment["run_name"]
    result_dir = RUNS_ROOT / experiment["run_name"]
    complete, resume, stop_reason = _run_completion(experiment, config, CHECKPOINT_ROOT, amp_enabled=False)
    status_path = result_dir / "run_status.json"
    if complete:
        write_json_with_retry(status_path, {"state": "complete", "updated_at_utc": utc_now(), "stop_reason": stop_reason, "run_name": experiment["run_name"]})
        print(f"[{index}/4] skip complete: {experiment['run_name']}")
    else:
        if MAX_NEW_RUNS_PER_SESSION is not None and new_runs >= MAX_NEW_RUNS_PER_SESSION:
            break
        write_json_with_retry(status_path, {"state": "resuming" if resume else "running", "updated_at_utc": utc_now(), "run_name": experiment["run_name"]})
        command = [sys.executable, str(REPO_DIR / "scripts" / "train.py"), "--config", str(config_path), "--checkpoint-dir", str(checkpoint_dir), "--result-dir", str(result_dir)]
        if resume:
            command.extend(["--resume", str(checkpoint_dir / "last.pt")])
        environment = os.environ.copy()
        environment["PYTHONPATH"] = str(REPO_DIR)
        environment["PYTHONUNBUFFERED"] = "1"
        environment["PYTHONIOENCODING"] = "utf-8"
        environment["PYTHONUTF8"] = "1"
        local_subprocess_root = LOCAL_STAGE_ROOT / "subprocess_logs"
        local_log_path = local_subprocess_root / f"{result_dir.name}.log"
        telemetry_process, telemetry_handle = _start_telemetry(HARDWARE_ROOT / f"gpu_telemetry_{experiment['run_name']}.csv")
        try:
            return_code, output_tail = _run_subprocess_streaming(command, REPO_DIR, environment, result_dir, local_root=local_subprocess_root)
        except BaseException as error:
            log_tail = "" if not local_log_path.is_file() else "\n".join(local_log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-200:])
            failure = {"state": "interrupted" if isinstance(error, KeyboardInterrupt) else "failed", "updated_at_utc": utc_now(), "run_name": experiment["run_name"], "error_type": type(error).__name__, "error": str(error), "traceback": traceback.format_exc()[-12000:], "traceback_tail": log_tail, "log_file": str(local_log_path)}
            write_json_with_retry(result_dir / "training_failure.json", failure)
            write_json_with_retry(status_path, failure)
            raise
        finally:
            try:
                _stop_telemetry(telemetry_process, telemetry_handle)
            except Exception as telemetry_error:
                print(f"Warning: telemetry cleanup failed: {telemetry_error}")
        if return_code != 0:
            failure = {"state": "failed", "updated_at_utc": utc_now(), "run_name": experiment["run_name"], "return_code": return_code, "traceback_tail": output_tail, "log_file": "train_subprocess.log"}
            write_json_with_retry(result_dir / "training_failure.json", failure)
            write_json_with_retry(status_path, failure)
            raise RuntimeError(f"Training failed: {experiment['run_name']}\n{output_tail}")
        complete, _, stop_reason = _run_completion(experiment, config, CHECKPOINT_ROOT, amp_enabled=False)
        if not complete:
            failure = {"state": "failed", "updated_at_utc": utc_now(), "run_name": experiment["run_name"], "error": "Training exited without epoch-limit or early-stopping completion"}
            write_json_with_retry(result_dir / "training_failure.json", failure)
            write_json_with_retry(status_path, failure)
            raise RuntimeError(failure["error"])
        write_json_with_retry(status_path, {"state": "complete", "updated_at_utc": utc_now(), "stop_reason": stop_reason, "run_name": experiment["run_name"]})
        (result_dir / "training_failure.json").unlink(missing_ok=True)
        new_runs += 1
    partial = analyze_stage2(experiments, RUN_LOG_ROOT, CHECKPOINT_ROOT, ANALYSIS_ROOT)
    if not partial["macro"].empty:
        display(partial["macro"])
print("Local ResNet-50 Stage 2 training pass finished")

{'epoch': 1, 'train_loss': 1.0687578254213521, 'val_loss': 0.813079053401947, 'val_macro_auroc': 0.897751207837144, 'val_macro_ap': 0.23482106694734503, 'val_macro_f1_tuned': 0.3021719918621895, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.23482106694734503, 'threshold_min': 0.6605975031852722, 'threshold_mean': 0.8107694940907615, 'threshold_max': 0.9741454720497131, 'train_seconds': 271.50390459998744, 'val_seconds': 14.107717199978651, 'train_images_per_second': 46.960650598321415, 'val_images_per_second': 159.48717769898343, 'peak_gpu_memory_gb': 0.9849066734313965, 'peak_gpu_memory_allocated_gb': 0.9849066734313965, 'peak_gpu_memory_reserved_gb': 1.283203125}
{'epoch': 2, 'train_loss': 0.8233730054228914, 'val_loss': 0.7659563168419732, 'val_macro_auroc': 0.9079184795801104, 'val_macro_ap': 0.29051102476493273, 'val_macro_f1_tuned': 0.3563339977551209, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.29051102476493273, 'threshold_min': 0.6004958152770

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,resnet50_stage2v2_bce_raw_aug_auprc_s8515_extt...,resnet50,bce,weighted_bce,None,raw,complete,21,0.528484,0.562494,...,92cc3c907c84549ac0a37a22236221582fb30159587c45...,0.562494,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True


{'epoch': 1, 'train_loss': 0.7327423299480887, 'val_loss': 0.5641248875194126, 'val_macro_auroc': 0.8936772809354226, 'val_macro_ap': 0.2652993454443995, 'val_macro_f1_tuned': 0.32693172088704775, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.2652993454443995, 'threshold_min': 0.6483588218688965, 'threshold_mean': 0.7600596121379307, 'threshold_max': 0.9278726577758789, 'train_seconds': 317.64038399999845, 'val_seconds': 17.306001499993727, 'train_images_per_second': 40.13973235846505, 'val_images_per_second': 130.0127010852747, 'peak_gpu_memory_gb': 0.9849071502685547, 'peak_gpu_memory_allocated_gb': 0.9849071502685547, 'peak_gpu_memory_reserved_gb': 1.283203125}
{'epoch': 2, 'train_loss': 0.5688602761824926, 'val_loss': 0.49130362564987606, 'val_macro_auroc': 0.9240984708438883, 'val_macro_ap': 0.316299885514784, 'val_macro_f1_tuned': 0.37329900021419554, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.316299885514784, 'threshold_min': 0.6766043901443481

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,resnet50_stage2v2_bce_raw_aug_auprc_s8515_extt...,resnet50,bce,weighted_bce,NaN,raw,complete,21,0.528484,0.562494,...,92cc3c907c84549ac0a37a22236221582fb30159587c45...,0.562494,0.0,0.0,0.0,0.000000,0.00000,0.00000,0.000000,True
1,resnet50_stage2v2_focal_g05_raw_aug_auprc_s851...,resnet50,focal_g05,sigmoid_focal,0.5,raw,complete,27,0.545510,0.566034,...,b46acfc6ad99d8bf131e0a97b8f032cf00e658bece2e12...,0.494003,7.0,7.0,3.0,0.017025,0.00354,-0.00222,0.003001,False


{'epoch': 1, 'train_loss': 0.5186488178945056, 'val_loss': 0.3873311420546638, 'val_macro_auroc': 0.8981456083034328, 'val_macro_ap': 0.2647660600056164, 'val_macro_f1_tuned': 0.32594739732595673, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.2647660600056164, 'threshold_min': 0.5983713269233704, 'threshold_mean': 0.7007861009665898, 'threshold_max': 0.7979169487953186, 'train_seconds': 222.65716740000062, 'val_seconds': 12.625864999979967, 'train_images_per_second': 57.2629219570318, 'val_images_per_second': 178.20561205141746, 'peak_gpu_memory_gb': 0.9849071502685547, 'peak_gpu_memory_allocated_gb': 0.9849071502685547, 'peak_gpu_memory_reserved_gb': 1.283203125}
{'epoch': 2, 'train_loss': 0.41253485577246723, 'val_loss': 0.33660605447159875, 'val_macro_auroc': 0.9301094218408343, 'val_macro_ap': 0.33729602046638385, 'val_macro_f1_tuned': 0.3960917681894386, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.33729602046638385, 'threshold_min': 0.570921063423

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,resnet50_stage2v2_bce_raw_aug_auprc_s8515_extt...,resnet50,bce,weighted_bce,NaN,raw,complete,21,0.528484,0.562494,...,92cc3c907c84549ac0a37a22236221582fb30159587c45...,0.562494,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,resnet50_stage2v2_focal_g05_raw_aug_auprc_s851...,resnet50,focal_g05,sigmoid_focal,0.5,raw,complete,27,0.545510,0.566034,...,b46acfc6ad99d8bf131e0a97b8f032cf00e658bece2e12...,0.494003,7.0,7.0,3.0,0.017025,0.003540,-0.002220,0.003001,False
2,resnet50_stage2v2_focal_g10_raw_aug_auprc_s851...,resnet50,focal_g10,sigmoid_focal,1.0,raw,complete,13,0.516605,0.509871,...,42a19bb221112caf0750e572cf946e6ad032495d555c92...,0.480469,5.0,9.0,7.0,-0.011879,-0.052623,-0.004159,-0.017079,False


{'epoch': 1, 'train_loss': 0.38330514679001826, 'val_loss': 0.29906986282931436, 'val_macro_auroc': 0.9008112626824332, 'val_macro_ap': 0.2616718789719804, 'val_macro_f1_tuned': 0.3328470127514974, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.2616718789719804, 'threshold_min': 0.5788168907165527, 'threshold_mean': 0.6853055145059314, 'threshold_max': 0.8716364502906799, 'train_seconds': 227.13112130001537, 'val_seconds': 13.072716700000456, 'train_images_per_second': 56.134975810552376, 'val_images_per_second': 172.11418648733675, 'peak_gpu_memory_gb': 0.9849071502685547, 'peak_gpu_memory_allocated_gb': 0.9849071502685547, 'peak_gpu_memory_reserved_gb': 1.283203125}
{'epoch': 2, 'train_loss': 0.2790658653284989, 'val_loss': 0.25780731050835715, 'val_macro_auroc': 0.928096970904141, 'val_macro_ap': 0.3185827541254665, 'val_macro_f1_tuned': 0.38122202631177776, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.3185827541254665, 'threshold_min': 0.613687634468

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,resnet50_stage2v2_bce_raw_aug_auprc_s8515_extt...,resnet50,bce,weighted_bce,NaN,raw,complete,21,0.528484,0.562494,...,92cc3c907c84549ac0a37a22236221582fb30159587c45...,0.562494,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,resnet50_stage2v2_focal_g05_raw_aug_auprc_s851...,resnet50,focal_g05,sigmoid_focal,0.5,raw,complete,27,0.545510,0.566034,...,b46acfc6ad99d8bf131e0a97b8f032cf00e658bece2e12...,0.494003,7.0,7.0,3.0,0.017025,0.003540,-0.002220,0.003001,False
2,resnet50_stage2v2_focal_g10_raw_aug_auprc_s851...,resnet50,focal_g10,sigmoid_focal,1.0,raw,complete,13,0.516605,0.509871,...,42a19bb221112caf0750e572cf946e6ad032495d555c92...,0.480469,5.0,9.0,7.0,-0.011879,-0.052623,-0.004159,-0.017079,False
3,resnet50_stage2v2_focal_g15_raw_aug_auprc_s851...,resnet50,focal_g15,sigmoid_focal,1.5,raw,complete,11,0.500705,0.503755,...,06481b639737de804fe1d44ca1ceaabd6795fba8f41202...,0.461350,5.0,9.0,5.0,-0.027779,-0.058739,-0.004793,-0.027007,False


Local ResNet-50 Stage 2 training pass finished


In [5]:
import shutil, zipfile
analysis = analyze_stage2(experiments, RUN_LOG_ROOT, CHECKPOINT_ROOT, ANALYSIS_ROOT)
display(analysis["macro"])
display(analysis["ranking"])
manifest = {"created_at_utc": utc_now(), "stage_id": LOCAL_STAGE_ID, "runtime_mode": "local_windows", "precision": "float32", "model": MODEL_NAME, "checkpoint_root": str(CHECKPOINT_ROOT), "run_log_root": str(RUN_LOG_ROOT), "selection_metric": "macro_average_precision", "external_test_used": False}
write_json_with_retry(RUN_LOG_ROOT / "experiment_manifest.json", manifest)
archive_base = EXPORT_PATH.with_suffix("")
if EXPORT_PATH.exists():
    EXPORT_PATH.unlink()
shutil.make_archive(str(archive_base), "zip", root_dir=LOCAL_STAGE_ROOT)
with zipfile.ZipFile(EXPORT_PATH) as archive:
    validate_export_members(archive.namelist())
print("Export ready:", EXPORT_PATH)
print("External test locked.")

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,resnet50_stage2v2_bce_raw_aug_auprc_s8515_extt...,resnet50,bce,weighted_bce,NaN,raw,complete,21,0.528484,0.562494,...,92cc3c907c84549ac0a37a22236221582fb30159587c45...,0.562494,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,resnet50_stage2v2_focal_g05_raw_aug_auprc_s851...,resnet50,focal_g05,sigmoid_focal,0.5,raw,complete,27,0.545510,0.566034,...,b46acfc6ad99d8bf131e0a97b8f032cf00e658bece2e12...,0.494003,7.0,7.0,3.0,0.017025,0.003540,-0.002220,0.003001,False
2,resnet50_stage2v2_focal_g10_raw_aug_auprc_s851...,resnet50,focal_g10,sigmoid_focal,1.0,raw,complete,13,0.516605,0.509871,...,42a19bb221112caf0750e572cf946e6ad032495d555c92...,0.480469,5.0,9.0,7.0,-0.011879,-0.052623,-0.004159,-0.017079,False
3,resnet50_stage2v2_focal_g15_raw_aug_auprc_s851...,resnet50,focal_g15,sigmoid_focal,1.5,raw,complete,11,0.500705,0.503755,...,06481b639737de804fe1d44ca1ceaabd6795fba8f41202...,0.461350,5.0,9.0,5.0,-0.027779,-0.058739,-0.004793,-0.027007,False


,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible,variant_order,rank
0,resnet50_stage2v2_bce_raw_aug_auprc_s8515_extt...,resnet50,bce,weighted_bce,NaN,raw,complete,21,0.528484,0.562494,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True,0,1
1,resnet50_stage2v2_focal_g05_raw_aug_auprc_s851...,resnet50,focal_g05,sigmoid_focal,0.5,raw,complete,27,0.545510,0.566034,...,7.0,7.0,3.0,0.017025,0.003540,-0.002220,0.003001,False,1,2
2,resnet50_stage2v2_focal_g10_raw_aug_auprc_s851...,resnet50,focal_g10,sigmoid_focal,1.0,raw,complete,13,0.516605,0.509871,...,5.0,9.0,7.0,-0.011879,-0.052623,-0.004159,-0.017079,False,2,3
3,resnet50_stage2v2_focal_g15_raw_aug_auprc_s851...,resnet50,focal_g15,sigmoid_focal,1.5,raw,complete,11,0.500705,0.503755,...,5.0,9.0,5.0,-0.027779,-0.058739,-0.004793,-0.027007,False,3,4


Export ready: D:\Junior\ĐATH\main\outputs\exports\04_2_local_resnet50_stage2_focalloss_v2_fp32_results.zip
External test locked.
